# Tflex testing
> Try to test tflex

In [ ]:
#| default_exp tflex_test

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#| export
import sys
from pathlib import Path
import socket
import subprocess
import polars as pl

In [ ]:
host =socket.gethostname()

In [ ]:
#| export
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(CV_TOOLS))


In [ ]:
#| export
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
sys.path.append(str(custom_lib_path))


In [ ]:
#| export
from cv_tools.imports import *
from cv_tools.core import *
from cv_tools.data_processing.smb_tools import *
from dotenv import load_dotenv


In [ ]:
#| export
if system() != "Linux" and socket.gethostname() =="ISCN5CG43344Q5":
    dotenv_path = Path(Path.cwd().parent , '.env')
else:
    dotenv_path = Path(Path.cwd().parent , '.env')
load_dotenv(dotenv_path=dotenv_path)

True

In [ ]:
USER,PWD = get_user_name_password(dotenv_path=dotenv_path)
USER

'WARaiwarupdada'

In [ ]:
#| export
def exec_shell(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.stdout



In [ ]:
def get_smb_path(
    p="",
    USER=USER, 
    PWD=PWD, 
    SERVER='SERVER', 
    SHARE='SHARE', 
    SUBPATH='SUBPATH'):
    """
    List files/directories on SMB share using smbclient command.
    
    Args:
        p (str): Path pattern to search for (supports wildcards)
        
    Returns:
        list: List of parsed file/directory entries, each as a list of fields
              (typically [name, attributes, size, date, time])
    
    Process:
    1. Builds smbclient command with server, share, credentials
    2. Executes 'ls' command with the given path pattern
    3. Filters out empty lines, hidden files (starting with '.'), and status messages
    4. Splits each line by multiple spaces to separate fields
    """
    # Build smbclient command with connection details and ls command
    cmd = ["smbclient", f"//{SERVER}/{SHARE}", f"--user", f"infineon/{USER}", f"--password", PWD]
   
    cmd += [f"-c", f"ls {SUBPATH}//{p}*"]
    
    
    # Execute command and split output into lines
    res = exec_shell(cmd).split("\n")
   
    
    # Filter out empty lines, hidden files, and status messages
    res = [x.strip() for x in res if len(x.strip()) and x.strip()[0] not in "." and not "blocks available" in x]
  
   
    # Parse each line by splitting on multiple spaces (smbclient output format)
    res = [re.split("\s\s\s*", x) for x in res]
    
    return res

In [ ]:
#SERVER='MUCSDV040.infineon.com'
SERVER="WARSDV010.infineon.com"
#SHARE='SiMic_MOP'
SHARE='AD_Easy_Panda'
#SUBPATH='01_public//01_microphone-issues//01_RN25_EB-tear-issue'
SUBPATH='Equipment/HCPA1/52714'
res_ =get_smb_path(
    p="",
    USER=USER, 
    PWD=PWD, 
    SERVER=SERVER, 
    SHARE=SHARE, 
    SUBPATH=SUBPATH
)

In [ ]:
def get_rec(
    SERVER, 
    SHARE, 
    SUBPATH,        
    path="", data=None, depth=0, max_depth=5, file_filter=None):
    """
    Recursively traverse SMB directory structure to find specific files.
    
    This function performs a depth-first search through SMB directories,
    collecting paths based on flexible filtering criteria.
    
    Args:
        path (str): Current directory path relative to SUBPATH
        data (list): Accumulator list for results (mutable default avoided)
        depth (int): Current recursion depth
        max_depth (int): Maximum recursion depth to prevent infinite loops
        file_filter (str, list, or callable): Filter for files to collect:
            - 'csv': Only CSV files
            - 'bmp': Only BMP files  
            - 'images': All image files (JPG, JPEG, PNG, BMP)
            - list: Specific filenames to match
            - callable: Custom function that takes filename and returns bool
            - None: All image files (default behavior)
    
    Returns:
        list: Paths to matching files and empty directories
        
    Examples:
        get_rec(file_filter='csv')  # Only CSV files
        get_rec(file_filter='bmp')  # Only BMP files
        get_rec(file_filter=['file1.txt', 'file2.csv'])  # Specific files
        get_rec(file_filter=lambda f: f.endswith('.log'))  # Custom filter
    """
    if data is None:
        data = []
    
    # Safety check for maximum recursion depth
    if depth >= max_depth:
        return data
    
    try:
        # Get directory listing from SMB share
        res = get_smb_path(p=path, SERVER=SERVER,SHARE=SHARE,SUBPATH=SUBPATH)
    except Exception as e:
        print(f"Error accessing SMB path '{path}': {e}")
        return data
    
    # Define file matching function based on filter type
    def matches_filter(filename):
        if file_filter is None or file_filter == 'images':
            # Default: all image files
            return any(filename.upper().endswith(ext) for ext in ['.JPG', '.JPEG', '.PNG', '.BMP'])
        elif file_filter == 'csv':
            return filename.upper().endswith('.CSV')
        elif file_filter == 'bmp':
            return filename.upper().endswith('.BMP')
        elif isinstance(file_filter, list):
            # Specific list of filenames
            return filename in file_filter
        elif callable(file_filter):
            # Custom filter function
            return file_filter(filename)
        else:
            # Fallback to string extension matching
            return filename.upper().endswith(f'.{file_filter.upper()}')
    
    # Process each entry in the directory
    for entry in res:
        if len(entry) < 2:  # Skip malformed entries
            continue
            
        filename, file_type = entry[0], entry[1]
        
        # Handle directories (marked with 'D')
        if file_type == "D":
            # Construct proper SMB path with forward slashes
            subdir_path = f"{path}/{filename}".strip("/")
            
            # Recursively explore subdirectory
            subdir_data = get_rec(data=[], depth=depth + 1, max_depth=max_depth, file_filter=file_filter, SERVER=SERVER,SHARE=SHARE,SUBPATH=SUBPATH)
            
            # If subdirectory has content, add it to results
            if subdir_data:
                data.extend(subdir_data)
            else:
                # Mark empty directory
                data.append(f"{path}/{filename}".strip("/"))
                
        # Handle files based on filter criteria
        elif matches_filter(filename):
            file_path = f"{path}/{filename}".strip("/")
            data.append(file_path)
    
    return list(set(data)) # remove duplicates


In [ ]:
SUBPATH

'Equipment/HCPA1/52714'

In [ ]:
data_ = get_rec(
SERVER=SERVER,
SHARE=SHARE,
SUBPATH=SUBPATH,
)

In [ ]:
data_

['intCycle1_3042400444552714_1.bmp',
 'intCycle1_3042400541552714_1.bmp',
 'intCycle1_3042400548552714_1.bmp',
 'intCycle1_3042403179552714_1.bmp',
 'intCycle1_3042403186552714_1.bmp',
 'intCycle1_3042400447552714_1.bmp',
 'intCycle1_3042400661552714_1.bmp',
 'intCycle1_3042400664552714_1.bmp',
 'intCycle1_3042400679552714_1.bmp',
 'intCycle1_3042400708552714_1.bmp',
 'intCycle1_3042400713552714_1.bmp',
 'intCycle1_3042400720552714_1.bmp',
 'intCycle1_3042403237552714_1.bmp',
 'intCycle1_3042403240552714_1.bmp',
 'intCycle1_3042403254552714_1.bmp',
 'intCycle1_3042400017552714_1.bmp',
 'intCycle1_3042400031552714_1.bmp',
 'intCycle1_3042400034552714_1.bmp',
 'intCycle1_3042400041552714_1.bmp',
 'intCycle1_3042400626552714_1.bmp',
 'intCycle1_3042400649552714_1.bmp',
 'intCycle1_3042400656552714_1.bmp',
 'intCycle1_3042400687552714_1.bmp',
 'intCycle1_3042400693552714_1.bmp',
 'intCycle1_3042400705552714_1.bmp',
 'intCycle1_3042403088552714_1.bmp',
 'intCycle1_3042403098552714_1.bmp',
 

In [ ]:
image_list_path = Path(r'/home/i4a_dev.work/data/projects/easy_front/DE_task/hochstrom_pin/vanessa_list')
fn = 'Matched_Picture_Rows_xx14.xlsx'
full_fn = pl.read_excel(Path(image_list_path, fn))
full_fn['ImageName'].unique().to_list()

Identifier,OrderNo(FAUF),ImageName,Productname,Path
i64,i64,str,str,str
82501410552,81960688,"""intCycle1_0082501405552714_2.b…","""404_F4""","""\Equipment\HCPA1\52714\intCycl…"
82502318552,81960688,"""intCycle1_0082502311552714_2.b…","""404_F4""","""\Equipment\HCPA1\52714\intCycl…"
82504513552,81960688,"""intCycle1_0082504137552714_2.b…","""404_F4""","""\Equipment\HCPA1\52714\intCycl…"
82506931552,81959750,"""intCycle1_0082506933552714_2.b…","""404_F4""","""\Equipment\HCPA1\52714\intCycl…"
82506934552,81959750,"""intCycle1_0082506933552714_2.b…","""404_F4""","""\Equipment\HCPA1\52714\intCycl…"
…,…,…,…,…
902400265552,81928415,"""intCycle1_3042404600552714_1.b…","""404_F4""","""\Equipment\HCPA1\52714\intCycl…"
42401797552,81928415,"""intCycle1_3042404600552714_1.b…","""404_F4""","""\Equipment\HCPA1\52714\intCycl…"
42404718552,81928415,"""intCycle1_3042404600552714_1.b…","""404_F4""","""\Equipment\HCPA1\52714\intCycl…"


In [ ]:
new_df = full_fn.with_columns(
    [
        pl.col('Path').cast(pl.Utf8).str.split("\\").list.get(2).alias("EQP"),
         pl.col('Path').cast(pl.Utf8).str.split("\\").list.get(3).alias("MAT")
    ]
    
                    )


In [ ]:
fl_list = new_df['ImageName'].unique().to_list()

In [ ]:
data_=None

In [ ]:
len(data_)

100348

In [ ]:
data_ = get_rec(
SERVER=SERVER,
SHARE=SHARE,
SUBPATH=SUBPATH,
    file_filter=fl_list
)

In [ ]:
PWD

'SsQcMU8IKAPEE4T9ZSsh'

In [ ]:
USER

'WARaiwarupdada'

In [ ]:
cmd_f

'mkdir -p /home/i4a_dev.work/data/projects/easy_front/DE_task/hochstrom_pin/failed_images; cd /home/i4a_dev.work/data/projects/easy_front/DE_task/hochstrom_pin/failed_images; smbget --update --password=$PWD --user=infineon/$USER --recursive "smb://WARSDV010.infineon.com/AD_Easy_Panda/Equipment/HCPA1/52714"'

In [ ]:
dest_path = Path(r'/home/i4a_dev.work/data/projects/easy_front/DE_task/hochstrom_pin/failed_images')

cmd_f = f'mkdir -p {dest_path}; cd {dest_path}; smbget --update --password=$SPCEIAL_ACCOUNT_UPD_ADA_PWD --user=infineon/$SPECIAL_ACCOUNT_UPD_ADA --recursive "smb://{SERVER}/{SHARE}/{SUBPATH}'
with open("workloads.txt", "w") as f:
    for item in data_:
        if Path(item).suffix == '.bmp':
            p = (item.strip("\\").replace("\\\\", "/"))
            NAME=p
            cmd_str=cmd_f +f'/{NAME}"'
            f.write(cmd_str + "\n")

In [ ]:
mkdir -p /home/i4a_dev.work/data/projects/easy_front/DE_task/hochstrom_pin/failed_images; cd /home/i4a_dev.work/data/projects/easy_front/DE_task/hochstrom_pin/failed_images; smbget --update --password=$PWD --user=infineon/$USER --recursive "smb://WARSDV010.infineon.com/AD_Easy_Panda/Equipment/HCPA1/52714/intCycle1_3472407315552714_2.bmp"

In [ ]:
len(data_)

2264

In [ ]:
set(data_)

{'81919414',
 '81919454',
 'intCycle1_0082501405552714_2.bmp',
 'intCycle1_0082502311552714_2.bmp',
 'intCycle1_0082504137552714_2.bmp',
 'intCycle1_0082506933552714_2.bmp',
 'intCycle1_0082508077552714_2.bmp',
 'intCycle1_0082509387552714_2.bmp',
 'intCycle1_2902400086552714_1.bmp',
 'intCycle1_2902400103552714_1.bmp',
 'intCycle1_2902400115552714_1.bmp',
 'intCycle1_2902400262552714_1.bmp',
 'intCycle1_2902400280552714_1.bmp',
 'intCycle1_2902400287552714_1.bmp',
 'intCycle1_2902400297552714_1.bmp',
 'intCycle1_2902400338552714_1.bmp',
 'intCycle1_2902400349552714_1.bmp',
 'intCycle1_2902400360552714_1.bmp',
 'intCycle1_3042400198552714_1.bmp',
 'intCycle1_3042400248552714_1.bmp',
 'intCycle1_3042400264552714_1.bmp',
 'intCycle1_3042400273552714_1.bmp',
 'intCycle1_3042400302552714_1.bmp',
 'intCycle1_3042400391552714_1.bmp',
 'intCycle1_3042401143552714_1.bmp',
 'intCycle1_3042401147552714_1.bmp',
 'intCycle1_3042401463552714_1.bmp',
 'intCycle1_3042401477552714_1.bmp',
 'intCycle1_

In [ ]:
len(fl_list)

72

In [ ]:
#| export
CURRETNT_NB='/home/ai_sintercra/homes/hasan/projects/git_data/new_test/nbs'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('50_tflex_test.ipynb')